# ASRA v0.1 — Phase 1 ARC-AGI-3 Notebook

**Goal:** Build a transition-centric adaptive reasoning agent for ARC-AGI-3.

This notebook focuses only on ARC-style interactive reasoning:

- ARC-AGI-3 environment interaction
- transition logs
- state graph
- action semantics inference
- exploration traces
- replay system
- simple planning / action selection

No external internet calls. No domain-specific datasets.

## 1. Configuration

Set `USE_MOCK_ENV = False` inside Kaggle after connecting the official ARC-AGI-3 competition environment / SDK.

This notebook includes a small mock grid environment so the ASRA pipeline can be tested end-to-end even before the official SDK adapter is finalized.

In [ ]:
import os
import json
import time
import math
import random
import hashlib
import statistics
from dataclasses import dataclass, asdict, field
from collections import defaultdict, Counter, deque
from typing import Any, Dict, List, Tuple, Optional

import numpy as np

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

USE_MOCK_ENV = True  # Change to False inside Kaggle when official ARC-AGI-3 environment is available.
MAX_EPISODES = 20
MAX_STEPS_PER_EPISODE = 80
ACTION_NAMES = ["ACTION1", "ACTION2", "ACTION3", "ACTION4", "ACTION5"]

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
TRANSITION_LOG_PATH = os.path.join(OUTPUT_DIR, "asra_v0_1_transition_log.jsonl")
STATE_GRAPH_PATH = os.path.join(OUTPUT_DIR, "asra_v0_1_state_graph.json")
REPLAY_PATH = os.path.join(OUTPUT_DIR, "asra_v0_1_replay.json")
SUMMARY_PATH = os.path.join(OUTPUT_DIR, "asra_v0_1_summary.json")

print("OUTPUT_DIR:", OUTPUT_DIR)

## 2. Core State Utilities

ASRA v0.1 treats every interaction as a scientific observation:

```text
state_t + action_t → state_{t+1}
```

The notebook hashes states, computes cell-level diffs, and stores transitions.

In [ ]:
def canonical_grid(grid: Any) -> List[List[int]]:
    arr = np.array(grid, dtype=int)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D grid, got shape {arr.shape}")
    return arr.tolist()

def state_hash(grid: Any) -> str:
    g = canonical_grid(grid)
    payload = json.dumps(g, separators=(",", ":"), sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def grid_diff(before: Any, after: Any) -> Dict[str, Any]:
    b = np.array(before, dtype=int)
    a = np.array(after, dtype=int)
    if b.shape != a.shape:
        return {
            "shape_changed": True,
            "before_shape": list(b.shape),
            "after_shape": list(a.shape),
            "num_changed_cells": None,
            "changed_cells": []
        }
    changed = []
    ys, xs = np.where(b != a)
    for y, x in zip(ys.tolist(), xs.tolist()):
        changed.append({
            "y": int(y),
            "x": int(x),
            "from": int(b[y, x]),
            "to": int(a[y, x])
        })
    return {
        "shape_changed": False,
        "before_shape": list(b.shape),
        "after_shape": list(a.shape),
        "num_changed_cells": len(changed),
        "change_ratio": float(len(changed) / b.size) if b.size else 0.0,
        "changed_cells": changed
    }

def summarize_grid(grid: Any) -> Dict[str, Any]:
    arr = np.array(grid, dtype=int)
    values, counts = np.unique(arr, return_counts=True)
    return {
        "shape": list(arr.shape),
        "color_counts": {str(int(v)): int(c) for v, c in zip(values, counts)},
        "nonzero_count": int(np.count_nonzero(arr)),
        "hash": state_hash(arr.tolist())
    }

def print_grid(grid: Any):
    arr = np.array(grid, dtype=int)
    for row in arr.tolist():
        print(" ".join(f"{v:2d}" for v in row))

## 3. Transition Records and State Graph

The transition log is the primary ASRA memory.  
The state graph aggregates repeated transitions into a compact world model.

In [ ]:
@dataclass
class TransitionRecord:
    episode_id: int
    step_id: int
    action: str
    state_hash: str
    next_state_hash: str
    state: List[List[int]]
    next_state: List[List[int]]
    diff: Dict[str, Any]
    reward: float = 0.0
    done: bool = False
    info: Dict[str, Any] = field(default_factory=dict)
    timestamp: float = field(default_factory=time.time)

class TransitionStore:
    def __init__(self):
        self.records: List[TransitionRecord] = []
        self.by_action: Dict[str, List[TransitionRecord]] = defaultdict(list)
        self.by_state_action: Dict[Tuple[str, str], List[TransitionRecord]] = defaultdict(list)
        self.hash_to_grid: Dict[str, List[List[int]]] = {}

    def add(self, record: TransitionRecord):
        self.records.append(record)
        self.by_action[record.action].append(record)
        self.by_state_action[(record.state_hash, record.action)].append(record)
        self.hash_to_grid[record.state_hash] = record.state
        self.hash_to_grid[record.next_state_hash] = record.next_state

    def save_jsonl(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            for r in self.records:
                f.write(json.dumps(asdict(r), separators=(",", ":")) + "\n")

    def to_dataframe(self):
        rows = []
        for r in self.records:
            rows.append({
                "episode_id": r.episode_id,
                "step_id": r.step_id,
                "action": r.action,
                "state_hash": r.state_hash[:10],
                "next_state_hash": r.next_state_hash[:10],
                "num_changed_cells": r.diff.get("num_changed_cells"),
                "change_ratio": r.diff.get("change_ratio"),
                "reward": r.reward,
                "done": r.done
            })
        return pd.DataFrame(rows) if pd is not None else rows

class StateGraph:
    def __init__(self):
        self.nodes: Dict[str, Dict[str, Any]] = {}
        self.edges: Dict[Tuple[str, str, str], Dict[str, Any]] = {}

    def add_transition(self, r: TransitionRecord):
        self.nodes.setdefault(r.state_hash, {"hash": r.state_hash, "grid": r.state, "visits": 0})
        self.nodes.setdefault(r.next_state_hash, {"hash": r.next_state_hash, "grid": r.next_state, "visits": 0})
        self.nodes[r.state_hash]["visits"] += 1
        key = (r.state_hash, r.action, r.next_state_hash)
        if key not in self.edges:
            self.edges[key] = {
                "from": r.state_hash,
                "to": r.next_state_hash,
                "action": r.action,
                "count": 0,
                "rewards": [],
                "diff_summaries": []
            }
        self.edges[key]["count"] += 1
        self.edges[key]["rewards"].append(float(r.reward))
        self.edges[key]["diff_summaries"].append({
            "num_changed_cells": r.diff.get("num_changed_cells"),
            "change_ratio": r.diff.get("change_ratio")
        })

    def export(self) -> Dict[str, Any]:
        edges = []
        for e in self.edges.values():
            rewards = e["rewards"]
            diffs = e["diff_summaries"]
            edges.append({
                "from": e["from"],
                "to": e["to"],
                "action": e["action"],
                "count": e["count"],
                "avg_reward": float(sum(rewards) / len(rewards)) if rewards else 0.0,
                "avg_changed_cells": float(np.mean([d["num_changed_cells"] for d in diffs if d["num_changed_cells"] is not None])) if diffs else None
            })
        return {
            "num_nodes": len(self.nodes),
            "num_edges": len(edges),
            "nodes": list(self.nodes.values()),
            "edges": edges
        }

    def save(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self.export(), f, indent=2)

## 4. Action Semantics Inference

ASRA does not assume that `ACTION1`, `ACTION2`, etc. have known meanings.  
It infers approximate action semantics from repeated transition effects.

In [ ]:
class ActionSemanticsInferencer:
    def __init__(self):
        self.action_effects: Dict[str, List[Dict[str, Any]]] = defaultdict(list)

    def observe(self, record: TransitionRecord):
        d = record.diff
        effect = {
            "num_changed_cells": d.get("num_changed_cells"),
            "change_ratio": d.get("change_ratio"),
            "shape_changed": d.get("shape_changed", False),
            "changed_positions": [(c["y"], c["x"]) for c in d.get("changed_cells", [])],
            "color_transitions": [(c["from"], c["to"]) for c in d.get("changed_cells", [])],
            "reward": record.reward,
            "done": record.done
        }
        self.action_effects[record.action].append(effect)

    def infer_action_summary(self, action: str) -> Dict[str, Any]:
        effects = self.action_effects.get(action, [])
        if not effects:
            return {"action": action, "observations": 0, "hypothesis": "unknown"}

        changed_counts = [e["num_changed_cells"] for e in effects if e["num_changed_cells"] is not None]
        pos_counter = Counter()
        color_counter = Counter()
        for e in effects:
            pos_counter.update(e["changed_positions"])
            color_counter.update(e["color_transitions"])

        if changed_counts:
            mean_changed = float(np.mean(changed_counts))
            std_changed = float(np.std(changed_counts))
        else:
            mean_changed = None
            std_changed = None

        top_positions = pos_counter.most_common(5)
        top_colors = color_counter.most_common(5)

        if mean_changed == 0:
            hypothesis = "null/no-op or blocked action"
        elif mean_changed is not None and mean_changed <= 1.5 and top_positions:
            hypothesis = f"localized cell operation near {top_positions[0][0]}"
        elif mean_changed is not None and mean_changed > 1.5:
            hypothesis = "multi-cell transformation or movement"
        else:
            hypothesis = "unknown transformation"

        consistency = None
        if changed_counts:
            consistency = float(1.0 / (1.0 + std_changed))

        return {
            "action": action,
            "observations": len(effects),
            "mean_changed_cells": mean_changed,
            "std_changed_cells": std_changed,
            "consistency_score": consistency,
            "top_changed_positions": [{"position": list(p), "count": c} for p, c in top_positions],
            "top_color_transitions": [{"from": int(k[0]), "to": int(k[1]), "count": c} for k, c in top_colors],
            "hypothesis": hypothesis
        }

    def summary(self) -> Dict[str, Any]:
        return {a: self.infer_action_summary(a) for a in sorted(self.action_effects.keys())}

## 5. Exploration Policy

The v0.1 policy is intentionally simple and interpretable:

- prefer less-tested actions at the current state
- prefer actions with uncertain semantics
- avoid recent loops
- exploit actions with positive reward when discovered

In [ ]:
class ASRAExplorer:
    def __init__(self, action_names: List[str], recent_window: int = 20):
        self.action_names = action_names
        self.recent_states = deque(maxlen=recent_window)
        self.global_action_counts = Counter()
        self.state_action_counts = Counter()
        self.action_rewards = defaultdict(list)

    def update(self, record: TransitionRecord):
        self.recent_states.append(record.next_state_hash)
        self.global_action_counts[record.action] += 1
        self.state_action_counts[(record.state_hash, record.action)] += 1
        self.action_rewards[record.action].append(float(record.reward))

    def choose_action(self, state_hash_value: str, semantics: ActionSemanticsInferencer) -> str:
        scores = {}
        for action in self.action_names:
            local_count = self.state_action_counts[(state_hash_value, action)]
            global_count = self.global_action_counts[action]
            reward_values = self.action_rewards[action]
            mean_reward = float(np.mean(reward_values)) if reward_values else 0.0

            sem = semantics.infer_action_summary(action)
            consistency = sem.get("consistency_score")
            uncertainty_bonus = 1.0 if consistency is None else (1.0 - min(1.0, consistency))

            novelty_score = 1.0 / (1.0 + local_count)
            exploration_score = 0.35 / (1.0 + global_count)

            scores[action] = (
                2.0 * novelty_score +
                0.7 * uncertainty_bonus +
                0.5 * mean_reward +
                exploration_score +
                random.random() * 0.05
            )

        return max(scores.items(), key=lambda kv: kv[1])[0]

## 6. Mock ARC-Like Environment

This is only for pipeline validation.  
Inside Kaggle, use the official ARC-AGI-3 environment adapter in the next section.

In [ ]:
class MockARCAGI3Env:
    '''
    Tiny 3x3 interactive grid environment for testing ASRA plumbing.

    Hidden action semantics:
    ACTION1: cycle top-left cell
    ACTION2: cycle top-center cell
    ACTION3: cycle center cell
    ACTION4: move nonzero center value to bottom-center when possible
    ACTION5: reset one random nonzero cell to zero

    Done condition: top-left, top-center, and center are nonzero.
    '''
    def __init__(self, max_steps=50):
        self.max_steps = max_steps
        self.step_count = 0
        self.grid = None

    def reset(self):
        self.step_count = 0
        self.grid = np.zeros((3, 3), dtype=int)
        self.grid[1, 1] = 1
        return self.grid.tolist()

    def step(self, action: str):
        self.step_count += 1
        g = self.grid.copy()

        if action == "ACTION1":
            g[0, 0] = (g[0, 0] + 1) % 4
        elif action == "ACTION2":
            g[0, 1] = (g[0, 1] + 1) % 4
        elif action == "ACTION3":
            g[1, 1] = (g[1, 1] + 1) % 4
        elif action == "ACTION4":
            if g[1, 1] != 0:
                g[2, 1] = g[1, 1]
                g[1, 1] = 0
        elif action == "ACTION5":
            nz = list(zip(*np.where(g != 0)))
            if nz:
                y, x = random.choice(nz)
                g[y, x] = 0

        self.grid = g
        done = bool(g[0, 0] != 0 and g[0, 1] != 0 and g[1, 1] != 0)
        reward = 1.0 if done else 0.0
        if self.step_count >= self.max_steps:
            done = True
        return self.grid.tolist(), reward, done, {"step_count": self.step_count}

## 7. Official ARC-AGI-3 Adapter

Use this cell to connect the official competition environment.

Because competition SDK names may change, the adapter is isolated here. The rest of ASRA only needs:

```text
env.reset() -> observation_grid
env.step(action_name) -> next_grid, reward, done, info
```

In [ ]:
class ARCAGI3Adapter:
    '''
    Replace the internals of this class with the official Kaggle ARC-AGI-3 SDK calls.

    Required ASRA interface:
        reset() -> grid
        step(action_name) -> next_grid, reward, done, info

    Suggested mapping:
        ACTION1..ACTION5 should map to the official environment action IDs.
    '''
    def __init__(self):
        self.env = None
        self._load_official_env()

    def _load_official_env(self):
        # Example placeholder. Keep this isolated so the main agent does not change.
        #
        # Possible future pattern:
        #   from arc_agi_3 import make_env
        #   self.env = make_env()
        #
        # Or:
        #   import arcagi3
        #   self.env = arcagi3.Environment(...)
        #
        raise NotImplementedError(
            "Official ARC-AGI-3 SDK adapter must be connected here inside Kaggle."
        )

    def reset(self):
        obs = self.env.reset()
        return self._extract_grid(obs)

    def step(self, action_name: str):
        action_id = self._map_action(action_name)
        result = self.env.step(action_id)
        # Adjust unpacking if official SDK uses a different return format.
        next_obs, reward, done, info = result
        return self._extract_grid(next_obs), float(reward), bool(done), dict(info)

    def _map_action(self, action_name: str):
        mapping = {
            "ACTION1": 0,
            "ACTION2": 1,
            "ACTION3": 2,
            "ACTION4": 3,
            "ACTION5": 4,
        }
        return mapping[action_name]

    def _extract_grid(self, obs: Any) -> List[List[int]]:
        if isinstance(obs, dict):
            for key in ["grid", "state", "observation", "board"]:
                if key in obs:
                    return canonical_grid(obs[key])
        return canonical_grid(obs)

## 8. ASRA v0.1 Agent Runner

In [ ]:
class ASRAV01Agent:
    def __init__(self, action_names: List[str]):
        self.action_names = action_names
        self.store = TransitionStore()
        self.graph = StateGraph()
        self.semantics = ActionSemanticsInferencer()
        self.explorer = ASRAExplorer(action_names)
        self.episode_summaries = []

    def run_episode(self, env, episode_id: int, max_steps: int):
        state = canonical_grid(env.reset())
        episode_reward = 0.0
        trace = []

        for step_id in range(max_steps):
            s_hash = state_hash(state)
            action = self.explorer.choose_action(s_hash, self.semantics)
            next_state, reward, done, info = env.step(action)
            next_state = canonical_grid(next_state)

            record = TransitionRecord(
                episode_id=episode_id,
                step_id=step_id,
                action=action,
                state_hash=s_hash,
                next_state_hash=state_hash(next_state),
                state=state,
                next_state=next_state,
                diff=grid_diff(state, next_state),
                reward=float(reward),
                done=bool(done),
                info=info or {}
            )

            self.store.add(record)
            self.graph.add_transition(record)
            self.semantics.observe(record)
            self.explorer.update(record)

            trace.append({
                "step_id": step_id,
                "action": action,
                "state_hash": record.state_hash,
                "next_state_hash": record.next_state_hash,
                "reward": float(reward),
                "done": bool(done),
                "num_changed_cells": record.diff.get("num_changed_cells")
            })

            episode_reward += float(reward)
            state = next_state

            if done:
                break

        summary = {
            "episode_id": episode_id,
            "steps": len(trace),
            "total_reward": episode_reward,
            "done": bool(trace[-1]["done"]) if trace else False,
            "trace": trace
        }
        self.episode_summaries.append(summary)
        return summary

    def run(self, env, episodes: int, max_steps: int):
        summaries = []
        for ep in range(episodes):
            summaries.append(self.run_episode(env, ep, max_steps))
        return summaries

    def save_outputs(self):
        self.store.save_jsonl(TRANSITION_LOG_PATH)
        self.graph.save(STATE_GRAPH_PATH)
        replay = {
            "episodes": self.episode_summaries,
            "action_semantics": self.semantics.summary()
        }
        with open(REPLAY_PATH, "w", encoding="utf-8") as f:
            json.dump(replay, f, indent=2)

        graph_export = self.graph.export()
        summary = {
            "agent": "ASRA v0.1 Phase 1",
            "num_transitions": len(self.store.records),
            "num_states": graph_export["num_nodes"],
            "num_edges": graph_export["num_edges"],
            "action_counts": dict(self.explorer.global_action_counts),
            "action_semantics": self.semantics.summary(),
            "output_files": {
                "transition_log": TRANSITION_LOG_PATH,
                "state_graph": STATE_GRAPH_PATH,
                "replay": REPLAY_PATH
            }
        }
        with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        return summary

## 9. Run ASRA v0.1

In [ ]:
if USE_MOCK_ENV:
    env = MockARCAGI3Env(max_steps=MAX_STEPS_PER_EPISODE)
else:
    env = ARCAGI3Adapter()

agent = ASRAV01Agent(ACTION_NAMES)
episode_summaries = agent.run(env, episodes=MAX_EPISODES, max_steps=MAX_STEPS_PER_EPISODE)
summary = agent.save_outputs()

print(json.dumps({
    "num_transitions": summary["num_transitions"],
    "num_states": summary["num_states"],
    "num_edges": summary["num_edges"],
    "action_counts": summary["action_counts"]
}, indent=2))

## 10. Inspect Transition Logs

In [ ]:
df = agent.store.to_dataframe()
if pd is not None:
    display(df.head(20))
else:
    print(df[:20])

## 11. Action Semantics Summary

In [ ]:
sem_summary = agent.semantics.summary()
print(json.dumps(sem_summary, indent=2))

## 12. Replay Viewer

Use this to inspect what the agent did step-by-step.

In [ ]:
def replay_episode(agent: ASRAV01Agent, episode_id: int, limit: int = 10):
    records = [r for r in agent.store.records if r.episode_id == episode_id]
    for r in records[:limit]:
        print("=" * 60)
        print(f"Episode {r.episode_id}, Step {r.step_id}, Action {r.action}, Reward {r.reward}, Done {r.done}")
        print("Before:")
        print_grid(r.state)
        print("After:")
        print_grid(r.next_state)
        print("Diff:", json.dumps(r.diff, indent=2))

replay_episode(agent, episode_id=0, limit=8)

## 13. State Graph Summary

In [ ]:
graph_export = agent.graph.export()
print("Nodes:", graph_export["num_nodes"])
print("Edges:", graph_export["num_edges"])

top_edges = sorted(graph_export["edges"], key=lambda e: e["count"], reverse=True)[:10]
for e in top_edges:
    print({
        "from": e["from"][:8],
        "to": e["to"][:8],
        "action": e["action"],
        "count": e["count"],
        "avg_reward": e["avg_reward"],
        "avg_changed_cells": e["avg_changed_cells"]
    })

## 14. Exploration Trace Visualization

In [ ]:
if plt is not None and pd is not None and len(df) > 0:
    action_counts = df["action"].value_counts().sort_index()
    ax = action_counts.plot(kind="bar", title="ASRA v0.1 Action Counts")
    ax.set_xlabel("Action")
    ax.set_ylabel("Count")
    plt.show()

    rewards_by_episode = (
        df.groupby("episode_id")["reward"].sum()
        if "episode_id" in df.columns else None
    )
    if rewards_by_episode is not None:
        ax = rewards_by_episode.plot(kind="line", marker="o", title="Reward by Episode")
        ax.set_xlabel("Episode")
        ax.set_ylabel("Total reward")
        plt.show()
else:
    print("Matplotlib or pandas unavailable; skipping plots.")

## 15. Submission Output Files

The notebook writes these files into the Kaggle working directory:

```text
asra_v0_1_transition_log.jsonl
asra_v0_1_state_graph.json
asra_v0_1_replay.json
asra_v0_1_summary.json
```

For the official ARC-AGI-3 submission, connect the adapter in Section 7 to the official SDK and ensure the final output format follows the competition's current instructions.

In [ ]:
print("Saved files:")
for path in [TRANSITION_LOG_PATH, STATE_GRAPH_PATH, REPLAY_PATH, SUMMARY_PATH]:
    print(path, "exists=", os.path.exists(path), "size=", os.path.getsize(path) if os.path.exists(path) else None)

## 16. Final Notes

This is ASRA Phase 1 only:

- transition-centric memory
- action-effect learning
- state graph world model
- simple uncertainty-aware exploration
- replay and audit artifacts

Next improvements for a stronger ARC-AGI-3 submission:

1. Replace mock environment with official ARC-AGI-3 SDK adapter.
2. Add object-centric parsing for grids.
3. Add goal-state inference.
4. Add planner over the learned state graph.
5. Add reset/dead-end detection.
6. Add per-game strategy library from replayed transition motifs.